[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhavierc/NLP_taller_1/blob/main/notebook_taller_3_v2.ipynb)

> **Nota sobre Google Colab:** igual que en taller 1, este cuaderno está pensado para 2 GPU Tesla T4 (Kaggle). En Colab gratuito (1 GPU) cambia `requested_gpus=1` en el PASO 02 antes de ejecutar.

# Taller 3 — NER clínico en español con Transformers preentrenados (BERT/RoBERTa)

## Integrantes

**Carlos Javier Cepeda, David Salamanca, Jose Milciades Ordoñez**

## Objetivo

Resolvemos el mismo problema del taller 1 — reconocimiento de entidades clínicas (síntomas, enfermedades, procedimientos, fármacos y proteínas) sobre el corpus SPACCC en español — pero reemplazamos la arquitectura Bi-LSTM entrenada desde cero por **transformers preentrenados**, haciendo fine-tuning con la librería `transformers` de Hugging Face.

## Qué cambia respecto a taller 1

- **Modelo**: en vez de una Bi-LSTM con embeddings entrenados desde cero, usamos dos transformers preentrenados en español:
  - `dccuchile/bert-base-spanish-wwm-cased` (**BETO**, BERT genérico, dominio general).
  - `PlanTL-GOB-ES/bsc-bio-ehr-es` (RoBERTa preentrenado en textos **biomédicos y clínicos** en español; técnicamente es RoBERTa, no BERT literal, pero lo documentamos igual porque es la mejor opción de dominio disponible en español).
- **spaCy desaparece**: ya no necesitamos tokenización ni POS tagging externos — el propio tokenizer de cada transformer (subword) reemplaza ese rol.
- **Dos variantes por modelo** (en vez de `con_pos`/`sin_pos`): `frozen` (encoder congelado, solo se entrena el cabezal de clasificación — *feature extraction*) y `fine_tuned` (todos los pesos se actualizan). En total comparamos **2 modelos × 2 variantes × 3 semillas = 12 entrenamientos**.
- **Entrenamiento**: usamos `Trainer` de Hugging Face en vez de un bucle manual en PyTorch. `Trainer` reparte automáticamente el trabajo entre las 2 GPU cuando las detecta — no escribimos `DataParallel` a mano como en taller 1.
- **Qué se mantiene igual (para que los resultados sean comparables entre talleres)**: el mismo corpus SPACCC, la misma partición fija por documento (`split_seed=42`, 600/150/250 documentos), la misma política de resolución de solapes de anotaciones ("flat-longest"), las mismas 5 categorías clínicas, y las mismas 3 semillas de entrenamiento (42, 123, 2026).

## Aviso importante

Este cuaderno requiere GPU real para ejecutarse (idealmente 2× Tesla T4 en Kaggle) y **no se ha ejecutado todavía** — a diferencia de taller 1, las celdas no tienen salidas guardadas. Ejecuta las PASOs en orden; la celda de PASO 10 imprime la tabla comparativa final que debe pegarse en el informe.

### PASO 01 — Dependencias

Instalamos las bibliotecas que necesita este cuaderno: `transformers` (modelos preentrenados y `Trainer`), `datasets` y `accelerate` (requerido internamente por `Trainer`). A diferencia de taller 1, aquí **no instalamos spaCy**: los tokenizers de BERT/RoBERTa reemplazan la tokenización y el POS tagging. Las métricas de NER las calculamos con la misma función `entity_spans` de taller 1 (PASO 07), no con `evaluate`/`seqeval`. Se ejecuta una sola vez por sesión y requiere Internet habilitado.

In [ ]:
# PASO 01 — Dependencias (Internet activado; ejecutar una vez por sesion)
import os, sys, subprocess, importlib.util

def pip_install(*args):
    """Corre pip capturando su salida: si falla, imprime el error real de pip en
    vez de un CalledProcessError vacio (necesario porque -q oculta el traceback de
    fondo que explica CUAL paquete fallo y por que)."""
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f'pip install fallo (codigo {result.returncode}): {args}')

# Kaggle a veces trae un pip desactualizado que no sabe generar metadatos para
# paquetes "legacy" (setup.py sin pyproject.toml); actualizarlo primero evita el
# error "python setup.py egg_info did not run successfully".
pip_install('--upgrade', 'pip', 'setuptools', 'wheel')

packages = ['transformers>=5,<6', 'datasets>=5,<6', 'accelerate>=1.1',
            'pandas>=2,<3', 'pyarrow>=14', 'requests>=2.31', 'tqdm>=4.66']
if importlib.util.find_spec('torch') is None:
    packages.append('torch>=2.4,<3')
pip_install(*packages)
print('PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.')

### PASO 02 — Configuracion, GPU y reproducibilidad

Definimos los dos checkpoints preentrenados, las dos variantes (`frozen`/`fine_tuned`), las tres semillas y las mismas etiquetas BIO de taller 1 (para que el F1 final sea comparable). Comprobamos que Kaggle detecte las dos Tesla T4. A diferencia de taller 1, **no envolvemos el modelo en `DataParallel` a mano**: el `Trainer` de Hugging Face lo hace automaticamente cuando corre en un solo proceso y detecta mas de una GPU visible.

In [ ]:
# PASO 02 - Dos checkpoints, dos variantes, tres semillas
import os, json, time, random, math, hashlib, platform, traceback
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import requests, torch
from torch import nn

CFG = dict(
    split_seed=42, training_seeds=[42, 123, 2026], fraction=1.0, validation_fraction=0.20,
    max_length=256, batch_size=16, patience=3, weight_decay=0.01, warmup_steps=0.1,
    learning_rate={'frozen': 5e-4, 'fine_tuned': 2e-5},
    epochs={'frozen': 15, 'fine_tuned': 5},
    class_weights=True,
    checkpoints={'beto': 'dccuchile/bert-base-spanish-wwm-cased',
                 'clinical': 'PlanTL-GOB-ES/bsc-bio-ehr-es'},
    variants=['frozen', 'fine_tuned'],
    run_final_test=False, final_test_checkpoint=None, final_test_variant=None, final_test_seed=None,
    requested_gpus=2, allow_cpu_for_debug=False,
)
assert 0 < CFG['fraction'] <= 1
assert 0 < CFG['validation_fraction'] < 1, 'Usa validation_fraction=0.20 para reservar el 20 %.'
assert isinstance(CFG['split_seed'], int) and 0 <= CFG['split_seed'] < 2**32
assert (isinstance(CFG['training_seeds'], list) and len(CFG['training_seeds']) >= 2
        and all(type(s) is int and 0 <= s < 2**32 for s in CFG['training_seeds'])
        and len(set(CFG['training_seeds'])) == len(CFG['training_seeds'])), 'Usa al menos dos semillas enteras distintas.'
random.seed(CFG['split_seed']); np.random.seed(CFG['split_seed']); torch.manual_seed(CFG['split_seed'])
assert CFG['requested_gpus'] in (1, 2), 'requested_gpus debe ser 1 o 2.'
available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if available_gpus == 0 and CFG['allow_cpu_for_debug']:
    device = torch.device('cpu')
    print('AVISO: modo de depuracion CPU explicito; NO valida CUDA ni las dos T4.')
elif available_gpus < CFG['requested_gpus']:
    raise RuntimeError(f'PASO 02: se requieren {CFG["requested_gpus"]} GPU CUDA y se detectaron '
                       f'{available_gpus}. En Kaggle selecciona GPU T4 x2 y reinicia la sesion. '
                       'Para usar deliberadamente una sola GPU cambia requested_gpus=1.')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
    torch.cuda.manual_seed_all(CFG['split_seed'])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = ROOT / 'spaccc_transformers'
WORK.mkdir(parents=True, exist_ok=True)
RUN = WORK / (time.strftime('%Y%m%d_%H%M%S') + '_' + str(time.time_ns())[-6:])
RUN.mkdir()
CACHE = WORK / 'cache'; CACHE.mkdir(exist_ok=True)
LABELS = ['CHEMICAL', 'DISEASE', 'PROCEDURE', 'PROTEIN', 'SYMPTOM']
TAGS = ['O'] + [f'{prefix}-{label}' for label in LABELS for prefix in ('B', 'I')]
tag_to_ix = {tag: i for i, tag in enumerate(TAGS)}
id2label = {i: tag for i, tag in enumerate(TAGS)}
label2id = {tag: i for i, tag in enumerate(TAGS)}
IGNORE = -100  # Convencion de Hugging Face: posiciones que no se evaluan (padding, subwords de continuacion, tokens especiales).
START = time.perf_counter()
REPORT = {'config': CFG.copy(), 'run_dir': str(RUN), 'device': str(device),
          'gpu': {'available_count': available_gpus, 'requested': CFG['requested_gpus'],
                  'devices': [{'id': i, 'name': torch.cuda.get_device_name(i),
                               'total_memory_gb': torch.cuda.get_device_properties(i).total_memory / 2**30}
                              for i in range(available_gpus)]},
          'cpu': {'logical_cores': os.cpu_count(), 'machine': platform.machine()},
          'versions': {'python': platform.python_version(), 'torch': str(torch.__version__)}}
(RUN / 'config.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
print('PASO 02 OK:', json.dumps(REPORT, indent=2))

### PASO 03 - Descarga y auditoria de los datos

Reutilizamos exactamente el mismo procedimiento de taller 1: descargamos y dejamos en cache las anotaciones NER, los documentos completos y el conjunto de prueba desde Hugging Face (mismo corpus SPACCC). Comprobamos columnas, categorias, duplicacion de textos entre particiones y correspondencia entre cada offset y el texto original. El analisis exploratorio detallado (distribucion de clases, longitud de entidades, densidad por documento) ya se hizo en taller 1 y no se repite aqui; esta PASO solo audita que la descarga sea consistente antes de tokenizar con los transformers.

In [ ]:
# PASO 03 - Descarga, textos completos y auditoria de todas las anotaciones (igual que taller 1)
def download_parquet(repo, split, name):
    """Descarga el parquet `split` del dataset `repo` (Hugging Face) y lo guarda como
    `name` dentro de la carpeta de cache. Si el archivo ya existe en cache lo reutiliza
    sin volver a descargarlo. Reintenta la descarga hasta 3 veces y valida que el
    parquet descargado se pueda leer antes de reemplazar la cache."""
    path = CACHE / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
        print('Descargando:', name, flush=True)
        error = None
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=(15, 120))
                response.raise_for_status()
                tmp = path.with_suffix('.tmp')
                tmp.write_bytes(response.content)
                pd.read_parquet(tmp)  # No guardar una respuesta invalida como cache.
                tmp.replace(path)
                break
            except Exception as exc:
                error = exc
                print(f'Intento {attempt + 1}/3: {type(exc).__name__}: {exc}', flush=True)
        if not path.exists():
            raise RuntimeError('PASO 03: revisa Internet de Kaggle y comparte este error.') from error
    return pd.read_parquet(path)

t0 = time.perf_counter()
df_train = download_parquet('IEETA/SPACCC-Spanish-NER', 'train', 'annotations_train.parquet')
df_test = download_parquet('IEETA/SPACCC-Spanish-NER', 'test', 'annotations_test.parquet')
df_docs = download_parquet('IEETA/SPACCC-documents', 'train', 'documents.parquet')
def document_id(value):
    """Obtiene el identificador de un documento a partir de su nombre de archivo."""
    return Path(str(value)).stem.removeprefix('es-')
required = {'filename', 'label', 'start_span', 'end_span', 'text'}
for frame in (df_train, df_test):
    assert required <= set(frame.columns), 'Columnas de anotacion inesperadas.'
    assert not frame[list(required)].isnull().any().any(), 'Anotaciones con valores nulos.'
    frame['doc_id'] = frame.filename.map(document_id)
    assert set(frame.label) <= set(LABELS), 'Categorias nuevas: revisar TAGS.'
df_docs['doc_id'] = df_docs.filename.map(document_id)
assert not df_docs.doc_id.duplicated().any(), 'Identificadores de documento duplicados.'
assert df_docs.document.map(lambda x: isinstance(x, str) and bool(x)).all()
texts = dict(zip(df_docs.doc_id, df_docs.document))
official_train_ids = sorted(df_train.doc_id.unique())
official_test_ids = sorted(df_test.doc_id.unique())
assert not set(official_train_ids) & set(official_test_ids), 'Fuga: documentos en train y test.'
assert (set(official_train_ids) | set(official_test_ids)) <= texts.keys(), 'Faltan documentos completos.'
bad_offsets, quote_only_differences = [], []
for split, frame in [('train', df_train), ('test', df_test)]:
    for row in frame.itertuples(index=False):
        a, b = int(row.start_span), int(row.end_span)
        actual = texts[row.doc_id][a:b]
        detail = {'split': split, 'doc_id': row.doc_id, 'start': a, 'end': b}
        if not (0 <= a < b <= len(texts[row.doc_id])):
            bad_offsets.append(detail)
        elif actual != row.text:
            if actual.replace(chr(34), '') == row.text.replace(chr(34), ''):
                quote_only_differences.append(detail)
            else:
                bad_offsets.append(detail)
assert not bad_offsets, f'Offsets incompatibles: {len(bad_offsets)}. Ejemplos: {bad_offsets[:5]}'
fingerprint = lambda s: hashlib.sha256(' '.join(s.split()).encode()).hexdigest()
train_hashes = {fingerprint(texts[k]) for k in official_train_ids}
test_hashes = {fingerprint(texts[k]) for k in official_test_ids}
assert not train_hashes & test_hashes, 'Textos duplicados entre train y test: revisar particion.'
REPORT['data'] = {'train_documents': len(official_train_ids), 'test_documents': len(official_test_ids),
                  'train_annotations': len(df_train), 'test_annotations': len(df_test),
                  'offset_errors': len(bad_offsets), 'quote_only_differences': quote_only_differences,
                  'download_audit_seconds': time.perf_counter() - t0,
                  'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in CACHE.glob('*.parquet')}}
print('PASO 03 OK:', json.dumps(REPORT['data'], indent=2))

### PASO 04 - Particion fija por documento

Usamos exactamente la misma particion de taller 1: `split_seed=42`, division por documento completo (nunca por fila de entidad), 600 documentos de train y 150 de validacion. Guardamos la misma huella SHA-256 para verificar que los 12 entrenamientos (2 modelos x 2 variantes x 3 semillas) comparten exactamente esta particion, y que es comparable con la de taller 1.

In [ ]:
# PASO 04 - Particion unica compartida por los dos modelos y sus variantes (sin test)
rng = np.random.default_rng(CFG['split_seed'])
order = rng.permutation(official_train_ids).tolist()
n_sample = min(len(order), max(5, math.ceil(len(order) * CFG['fraction'])))
selected = order[:n_sample]
n_val = max(1, round(n_sample * CFG['validation_fraction']))
val_ids, train_ids = sorted(selected[:n_val]), sorted(selected[n_val:])
assert train_ids and val_ids
assert not set(train_ids) & set(val_ids)
assert not {fingerprint(texts[k]) for k in train_ids} & {fingerprint(texts[k]) for k in val_ids}
split_payload = {'train': train_ids, 'validation': val_ids, 'official_test': official_test_ids}
split_sha256 = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()
REPORT['split'] = {'sample_documents': n_sample, 'train_documents': len(train_ids),
                   'validation_documents': len(val_ids), 'test_used_for_training': False,
                   'split_seed': CFG['split_seed'], 'sha256': split_sha256}
(RUN / 'split.json').write_text(json.dumps({**split_payload, 'split_seed': CFG['split_seed'], 'sha256': split_sha256}, indent=2), encoding='utf-8')
print('PASO 04 - MUESTRA:', json.dumps(REPORT['split'], indent=2))
print('Anotaciones por categoria en la muestra:')
print(df_train[df_train.doc_id.isin(selected)].label.value_counts().to_string())
print('PASO 04 OK. Los 2 modelos x 2 variantes x 3 semillas usaran estos mismos documentos; todavia no comenzo a entrenar.')
TALLER1_SPLIT_SHA256 = 'ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497'
if split_sha256 == TALLER1_SPLIT_SHA256:
    print('Huella de particion identica a taller 1: los resultados son comparables entre talleres.')
else:
    print('AVISO: la huella de particion no coincide con la de taller 1 '
          f'(actual={split_sha256}, taller1={TALLER1_SPLIT_SHA256}). '
          'Revisa split_seed/fraction/validation_fraction si esperabas la misma partición; '
          'la comparación entre talleres dejaría de ser sobre el mismo conjunto de documentos.')

### PASO 05 - Tokenizacion subword y alineacion BIO por modelo

Reemplazamos la tokenizacion de spaCy por el tokenizer *fast* de cada transformer (BETO y el RoBERTa clinico tienen vocabularios distintos, asi que tokenizamos una vez por modelo). Para cada documento completo obtenemos los subwords y sus offsets de caracter, alineamos las anotaciones exigiendo coincidencia **estricta** de limites (misma politica que taller 1) y resolvemos solapes quedandonos con la entidad mas larga. Dividimos cada documento en bloques de a lo sumo `max_length` subwords sin cortar ninguna entidad a la mitad, igual que en taller 1. El resultado se cachea en disco por modelo.

In [ ]:
# PASO 05 - Tokenizacion subword y alineacion BIO (una vez por checkpoint)
from transformers import AutoTokenizer
from tqdm.auto import tqdm

t0 = time.perf_counter()
tokenizers = {key: AutoTokenizer.from_pretrained(checkpoint) for key, checkpoint in CFG['checkpoints'].items()}
for key, tok in tokenizers.items():
    assert tok.is_fast, f'{key}: se necesita un tokenizer fast (con offset_mapping) para alinear BIO por subword.'
    print(f'{key}: {CFG["checkpoints"][key]} -> vocab_size={tok.vocab_size}, fast={tok.is_fast}')

def resolve_gold(text, offset_mapping, subset):
    """Asigna una etiqueta BIO a cada subword de un documento a partir de sus offsets
    de caracter (inicio, fin) y las anotaciones de ese documento.

    Igual que en taller 1: exige alineacion ESTRICTA (el limite de la anotacion debe
    coincidir exactamente con el limite de algun subword) y, si dos anotaciones
    alineadas se solapan, conserva la mas larga (y, en empate, la que empieza antes).
    Los subwords que pertenecian a una anotacion descartada quedan en IGNORE en vez
    de 'O', salvo que otra entidad retenida ya les haya asignado una etiqueta valida.

    NOTA: el tokenizer del checkpoint 'clinical' (RoBERTa byte-level, estilo GPT-2)
    incluye el espacio previo dentro del offset del propio token (el de BETO, WordPiece,
    no tiene este problema). Sin recortar ese espacio inicial, casi ninguna entidad
    alinea porque las anotaciones de SPACCC empiezan en la primera letra, no en el
    espacio -- se verifico empiricamente contra el corpus real (96% quedaban como
    'unaligned' sin este ajuste, 0.3% con el)."""
    n = len(offset_mapping)
    starts, ends = {}, {}
    for idx, (s, e) in enumerate(offset_mapping):
        if s == e:
            continue
        content_start = s
        while content_start < e and text[content_start].isspace():
            content_start += 1
        starts.setdefault(content_start, idx)
        ends[e] = idx
    gold = [0] * n
    candidates, ignored_spans, seen = [], [], set()
    stats = Counter()
    for row in subset.itertuples(index=False):
        a, b, label = int(row.start_span), int(row.end_span), row.label
        stats['annotations'] += 1
        identity = (a, b, label)
        if identity in seen:
            stats['duplicates'] += 1
            continue
        seen.add(identity)
        if a not in starts or b not in ends or ends[b] < starts[a]:
            stats['unaligned'] += 1
            ignored_spans.append((a, b))
            continue
        candidates.append((a, b, label, starts[a], ends[b] + 1))
    for a, b, label, i, j in sorted(candidates, key=lambda x: (-(x[1] - x[0]), x[0], x[2])):
        if any(gold[k] != 0 for k in range(i, j)):
            stats['overlap_excluded'] += 1
            ignored_spans.append((a, b))
            continue
        gold[i] = tag_to_ix[f'B-{label}']
        gold[i + 1:j] = [tag_to_ix[f'I-{label}']] * (j - i - 1)
        stats['retained'] += 1
    for k, (s, e) in enumerate(offset_mapping):
        if gold[k] == 0 and any(s < b2 and e > a2 for a2, b2 in ignored_spans):
            gold[k] = IGNORE
    return gold, stats

def make_examples(checkpoint_key, ids, annotations, split_name):
    """Tokeniza los documentos `ids` con el tokenizer de `checkpoint_key`, alinea las
    anotaciones a nivel de subword y divide cada documento en bloques de a lo sumo
    `CFG['max_length']` subwords (incluyendo [CLS]/[SEP]) sin cortar ninguna entidad
    a la mitad -- misma logica de bloques de taller 1, aplicada a subwords en vez de
    tokens de spaCy. Cachea el resultado en disco (una cache distinta por checkpoint)."""
    tokenizer = tokenizers[checkpoint_key]
    subset_all = annotations[annotations.doc_id.isin(ids)]
    # 'policy' incluye una version del algoritmo de alineacion: al subirla invalidamos
    # la cache vieja automaticamente (necesario porque corregimos un bug de alineacion
    # que no cambiaba ids/checkpoint/textos/anotaciones, solo la logica de resolve_gold).
    key = hashlib.sha256(json.dumps({'ids': ids, 'max_length': CFG['max_length'],
        'checkpoint': CFG['checkpoints'][checkpoint_key],
        'policy': 'flat-longest-strict-strip-leading-space-v2',
        'texts': [fingerprint(texts[k]) for k in ids],
        'annotations': subset_all[['doc_id', 'start_span', 'end_span', 'label']].to_json()},
        sort_keys=True).encode()).hexdigest()
    cache_file = CACHE / f'features_{checkpoint_key}_{key}.json'
    start = time.perf_counter()
    if cache_file.exists():
        payload = json.loads(cache_file.read_text(encoding='utf-8'))
        payload['stats']['cache_hit'] = True
        payload['stats']['seconds_this_run'] = time.perf_counter() - start
        (RUN / f'alignment_{checkpoint_key}_{split_name}.json').write_text(json.dumps(payload['stats'], indent=2), encoding='utf-8')
        return payload['examples'], payload['stats']
    groups = {k: g for k, g in subset_all.groupby('doc_id')}
    empty_subset = subset_all.iloc[0:0]
    examples = []
    stats = Counter(documents=len(ids), annotations=0, duplicates=0, unaligned=0,
                    overlap_excluded=0, retained=0, tokens=0, ignored_tokens=0)
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    tokens_per_block = CFG['max_length'] - 2
    for doc_id in tqdm(ids, desc=f'Tokenizando {checkpoint_key} {split_name}'):
        encoding = tokenizer(texts[doc_id], add_special_tokens=False, return_offsets_mapping=True)
        input_ids, offset_mapping = encoding['input_ids'], encoding['offset_mapping']
        assert len(input_ids) > 0
        gold, doc_stats = resolve_gold(texts[doc_id], offset_mapping, groups.get(doc_id, empty_subset))
        for k, v in doc_stats.items():
            stats[k] += v
        n = len(input_ids)
        stats['tokens'] += n
        stats['ignored_tokens'] += gold.count(IGNORE)
        start_token = 0
        while start_token < n:
            end = min(start_token + tokens_per_block, n)
            # Si el siguiente subword es I-, el corte cae dentro de una entidad: retrocedemos
            # hasta su inicio (o, si eso vacia el bloque, avanzamos hasta que termine).
            if end < n and gold[end] >= 0 and TAGS[gold[end]].startswith('I-'):
                cut = end
                while cut > start_token and gold[cut] >= 0 and TAGS[gold[cut]].startswith('I-'):
                    cut -= 1
                if cut > start_token:
                    end = cut
                else:
                    while end < n and gold[end] >= 0 and TAGS[gold[end]].startswith('I-'):
                        end += 1
            assert end > start_token
            block_ids = [cls_id] + input_ids[start_token:end] + [sep_id]
            block_labels = [IGNORE] + gold[start_token:end] + [IGNORE]
            examples.append({'doc_id': doc_id, 'input_ids': block_ids,
                             'attention_mask': [1] * len(block_ids), 'labels': block_labels})
            start_token = end
    stats = dict(stats)
    stats.update(cache_hit=False, sequences=len(examples), seconds_this_run=time.perf_counter() - start)
    assert stats.get('retained', 0) > 0, 'No quedaron entidades. Comparte este error.'
    payload = {'examples': examples, 'stats': stats}
    cache_file.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    (RUN / f'alignment_{checkpoint_key}_{split_name}.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
    return examples, stats

train_examples, val_examples, REPORT['alignment'] = {}, {}, {}
for checkpoint_key in CFG['checkpoints']:
    train_examples[checkpoint_key], train_stats = make_examples(checkpoint_key, train_ids, df_train, 'train')
    val_examples[checkpoint_key], val_stats = make_examples(checkpoint_key, val_ids, df_train, 'validation')
    REPORT['alignment'][checkpoint_key] = {'train': train_stats, 'validation': val_stats}
    print(checkpoint_key, 'train', json.dumps(train_stats, indent=2))
    print(checkpoint_key, 'validation', json.dumps(val_stats, indent=2))
REPORT['preprocessing_seconds'] = time.perf_counter() - t0
print('PASO 05 OK. Bloques por modelo:', {k: len(v) for k, v in train_examples.items()})

### PASO 06 - Datasets, collator dinamico y pesos por clase

Convertimos los bloques de cada modelo en un `datasets.Dataset` de Hugging Face. El padding dinamico ya no lo escribimos a mano: `DataCollatorForTokenClassification` rellena `input_ids`/`attention_mask`/`labels` por lote automaticamente (con `label_pad_token_id=IGNORE`). Mantenemos los pesos por clase de taller 1 para compensar el desbalance hacia la etiqueta `O`; como `Trainer` no pondera clases por defecto, se los pasamos mediante `compute_loss_func`.

In [ ]:
# PASO 06 - Datasets de Hugging Face, collator dinamico y perdida ponderada por clase
from datasets import Dataset
from transformers import DataCollatorForTokenClassification

def make_compute_loss(w):
    """Cierra sobre los pesos de un modelo y devuelve la funcion de perdida que
    `Trainer(compute_loss_func=...)` llamara con (outputs, labels, num_items_in_batch).
    Es entropia cruzada ponderada por clase, ignorando las posiciones IGNORE
    (padding y subwords no evaluables) -- el equivalente de `loss_parts` en taller 1."""
    def compute_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=w.to(logits.device), ignore_index=IGNORE)
        return loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
    return compute_loss

hf_datasets, collators, class_weights, loss_fns = {}, {}, {}, {}
REPORT['vocabulary'] = {}
for checkpoint_key in CFG['checkpoints']:
    hf_datasets[checkpoint_key] = {
        'train': Dataset.from_list(train_examples[checkpoint_key]),
        'validation': Dataset.from_list(val_examples[checkpoint_key]),
    }
    collators[checkpoint_key] = DataCollatorForTokenClassification(
        tokenizer=tokenizers[checkpoint_key], label_pad_token_id=IGNORE)
    label_counts = Counter(y for e in train_examples[checkpoint_key] for y in e['labels'] if y != IGNORE)
    assert label_counts[0] > 0 and sum(v for k, v in label_counts.items() if k > 0) > 0
    weights = torch.ones(len(TAGS))
    if CFG['class_weights']:
        freq = torch.tensor([label_counts[i] for i in range(len(TAGS))], dtype=torch.float)
        present = freq > 0
        weights[present] = (freq[present].sum() / (present.sum() * freq[present])).sqrt().clamp(0.25, 5.0)
    class_weights[checkpoint_key] = weights
    loss_fns[checkpoint_key] = make_compute_loss(weights)
    REPORT['vocabulary'][checkpoint_key] = {
        'train_tag_counts': {TAGS[k]: v for k, v in sorted(label_counts.items())},
        'missing_train_tags': [TAGS[i] for i in range(len(TAGS)) if label_counts[i] == 0],
        'tokenizer_vocab_size': tokenizers[checkpoint_key].vocab_size,
        'train_blocks': len(train_examples[checkpoint_key]), 'validation_blocks': len(val_examples[checkpoint_key])}
    print(checkpoint_key, 'PASO 06 OK:', json.dumps(REPORT['vocabulary'][checkpoint_key], indent=2))
    if REPORT['vocabulary'][checkpoint_key]['missing_train_tags']:
        print(f'AVISO ({checkpoint_key}): faltan etiquetas en esta muestra; el test no permitira valorar esas etiquetas.')

### PASO 07 - Modelos, variantes y metricas de NER

Cargamos cada checkpoint con un cabezal de clasificacion de tokens (`AutoModelForTokenClassification`) del tamano de nuestras 11 etiquetas BIO. La variante `frozen` congela el encoder preentrenado y solo entrena el cabezal; `fine_tuned` deja todos los pesos entrenables. Para evaluar usamos `entity_spans`/`scores_from_counts`, la misma funcion que taller 1 implemento a mano (coincidencia exacta de limites y categoria) -- no `seqeval`: verificamos que su modo estricto descarta las transiciones BIO ilegales en vez de contarlas como prediccion incorrecta, lo que haria el F1 no comparable entre talleres.

In [ ]:
# PASO 07 - Modelos, variante congelada/fine-tuned y metricas de NER
from transformers import AutoModelForTokenClassification

def load_model(checkpoint_key):
    """Carga un modelo preentrenado con un cabezal de clasificacion de tokens del
    tamano de nuestras 11 etiquetas BIO. `id2label`/`label2id` hacen que las
    predicciones y el checkpoint guardado usen nombres de etiqueta, no solo indices."""
    return AutoModelForTokenClassification.from_pretrained(
        CFG['checkpoints'][checkpoint_key], num_labels=len(TAGS), id2label=id2label, label2id=label2id)

def apply_variant(model, variant):
    """Aplica la variante `frozen` (congela el encoder base; solo se entrena el
    cabezal de clasificacion) o `fine_tuned` (todos los pesos entrenables)."""
    if variant == 'frozen':
        for param in model.base_model.parameters():
            param.requires_grad = False
    elif variant != 'fine_tuned':
        raise ValueError(f'Variante desconocida: {variant}')
    return model

def entity_spans(tag_ids):
    """Agrupa una secuencia de indices de etiqueta BIO en entidades completas
    (inicio, fin, categoria) -- identica a la de taller 1 (y a la del taller 2 de
    Jose): una I- sin entidad activa de la misma categoria se trata como inicio de
    una entidad nueva (reparacion deterministica), contada aparte en `invalid`. La
    usamos en vez de `seqeval` para que el F1 sea estrictamente comparable entre los
    tres talleres: `seqeval` en modo estricto directamente DESCARTA una transicion
    BIO ilegal (no la cuenta como prediccion), mientras que `entity_spans` la cuenta
    como una entidad predicha (probablemente incorrecta) y penaliza la precision --
    se verifico esta diferencia de forma empirica antes de hacer este cambio."""
    result, active, begin, invalid = set(), None, None, 0
    for i, idx in enumerate(list(tag_ids) + [0]):
        tag = TAGS[idx] if idx >= 0 else 'O'
        prefix, label = tag.split('-', 1) if tag != 'O' else ('O', None)
        continuing = prefix == 'I' and label == active
        if active is not None and not continuing:
            result.add((begin, i, active)); active = None
        if prefix in ('B', 'I') and not continuing:
            invalid += int(prefix == 'I')
            begin, active = i, label
    return result, invalid

def scores_from_counts(tp, predicted, gold):
    """Calcula precision, recall y F1 a partir de conteos agregados: verdaderos
    positivos (tp), total de entidades predichas y total de entidades reales (gold)
    -- identica a la de taller 1."""
    precision = tp / predicted if predicted else 0.0
    recall = tp / gold if gold else 0.0
    return {'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.0,
            'support': gold, 'predicted': predicted}

def compute_metrics(eval_pred):
    """Convierte logits/etiquetas a indices BIO y aplica entity_spans/scores_from_counts
    (coincidencia EXACTA de limites y categoria) -- la misma funcion de taller 1, en vez
    de seqeval. Como las etiquetas ya traen -100 tanto en el padding final como en los
    subwords de continuacion, no hace falta truncar por longitud: basta con relabelar
    esas posiciones a 'O' antes de reconstruir las entidades (igual que en taller 1),
    preservando la posicion de cada token para no unir entidades a traves de un hueco
    ignorado. Devuelve F1/precision/recall globales y F1 por categoria clinica."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    totals = {label: Counter() for label in LABELS}
    invalid = 0
    correct = tokens = 0
    for pred_row, label_row in zip(predictions.tolist(), labels.tolist()):
        mask_valid = [y != IGNORE for y in label_row]
        correct += sum((pi == yi) and mv for pi, yi, mv in zip(pred_row, label_row, mask_valid))
        tokens += sum(mask_valid)
        p = [x if y != IGNORE else 0 for x, y in zip(pred_row, label_row)]
        g = [y if y != IGNORE else 0 for y in label_row]
        ps, errors = entity_spans(p); gs, gold_errors = entity_spans(g)
        assert gold_errors == 0, 'BIO gold invalido.'
        invalid += errors
        for label in LABELS:
            totals[label].update(tp=sum(x[2] == label for x in ps & gs),
                predicted=sum(x[2] == label for x in ps), gold=sum(x[2] == label for x in gs))
    per_label = {k: scores_from_counts(v['tp'], v['predicted'], v['gold']) for k, v in totals.items()}
    micro = scores_from_counts(sum(v['tp'] for v in totals.values()),
                               sum(v['predicted'] for v in totals.values()), sum(v['gold'] for v in totals.values()))
    metrics = {'precision': micro['precision'], 'recall': micro['recall'], 'f1': micro['f1'],
              'accuracy': correct / tokens if tokens else 0.0, 'invalid_bio_transitions': invalid}
    for label in LABELS:
        metrics[f'f1_{label}'] = per_label[label]['f1']
        metrics[f'support_{label}'] = per_label[label]['support']
    return metrics

print('PASO 07 OK. Modelos, variantes y metricas definidos; aun no se ha entrenado.')

### PASO 08 - Prueba tecnica de las cuatro combinaciones

Antes de invertir tiempo en los 12 entrenamientos, probamos cada combinacion (checkpoint x variante) con un modelo temporal: comprobamos que la perdida baja al repetir un lote pequeno, que los gradientes son finitos y que la variante `frozen` realmente deja el encoder sin gradiente (solo el cabezal se mueve). Si esta PASO falla, no continues con la PASO 09.

In [ ]:
# PASO 08 - PRUEBA TECNICA: gradientes finitos, la perdida baja, frozen congela el encoder
def run_smoke_test(checkpoint_key, variant):
    """Prueba tecnica rapida (no es el entrenamiento real) antes de invertir tiempo en
    las 12 corridas de la PASO 09. En un lote pequeno verifica que la perdida baja al
    repetir el mismo lote, que los gradientes de los parametros entrenables son
    finitos y que, en la variante `frozen`, el encoder no recibe gradiente. Los pesos
    de este modelo temporal se descartan; no pasan al entrenamiento real."""
    collator = collators[checkpoint_key]
    probe_examples = train_examples[checkpoint_key][:4]
    batch = collator([{k: v for k, v in e.items() if k != 'doc_id'} for e in probe_examples])
    batch = {k: v.to(device) for k, v in batch.items()}
    torch.manual_seed(CFG['training_seeds'][0])
    model = apply_variant(load_model(checkpoint_key), variant).to(device)
    trainable_before = {n for n, p in model.named_parameters() if p.requires_grad}
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.01)
    losses = []
    for _ in range(10):
        optimizer.zero_grad(set_to_none=True)
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        loss = loss_fns[checkpoint_key](outputs, batch['labels'])
        assert torch.isfinite(loss), 'Perdida no finita en prueba tecnica.'
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad), 1.0, error_if_nonfinite=True)
        assert norm > 0, 'Gradiente nulo: revisar etiquetas.'
        optimizer.step()
        losses.append(loss.item())
    assert min(losses[-3:]) < losses[0], 'La perdida no bajo al repetir el lote.'
    if variant == 'frozen':
        encoder_grad_free = all(p.grad is None for n, p in model.named_parameters()
                                if n not in trainable_before)
        assert encoder_grad_free, 'El encoder recibio gradiente en la variante frozen.'
    return {'passed': True, 'steps': len(losses), 'initial_loss': losses[0], 'final_loss': losses[-1],
            'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
            'total_parameters': sum(p.numel() for p in model.parameters())}

try:
    smoke_results = {}
    for checkpoint_key in CFG['checkpoints']:
        for variant in CFG['variants']:
            smoke_results[f'{checkpoint_key}_{variant}'] = run_smoke_test(checkpoint_key, variant)
    REPORT['smoke_test'] = {'passed': all(r['passed'] for r in smoke_results.values()), 'variants': smoke_results}
except Exception as exc:
    failure = {'status': 'ERROR', 'cell': '08', 'error': str(exc), 'traceback': traceback.format_exc()}
    (RUN / 'error.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
    print('ERROR_PARA_COMPARTIR', json.dumps(failure, indent=2), flush=True)
    raise
print('PASO 08 OK:', json.dumps(REPORT['smoke_test'], indent=2))

### PASO 09 - Doce entrenamientos: 2 modelos x 2 variantes x 3 semillas

Entrenamos las 12 combinaciones (`beto`/`clinical` x `frozen`/`fine_tuned` x semillas 42/123/2026) con `Trainer`. Cada corrida reinicia la semilla y carga pesos frescos del checkpoint preentrenado, entrena hasta el maximo de epocas de su variante o hasta parada temprana (`EarlyStoppingCallback`), selecciona el mejor checkpoint por F1 de validacion (`load_best_model_at_end`) y lo guarda en disco. Esta es la PASO que consume tiempo -- con 2 GPU, `Trainer` reparte cada lote entre ambas automaticamente.

In [ ]:
# PASO 09 - DOCE ENTRENAMIENTOS: 2 modelos x 2 variantes x 3 semillas
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

assert REPORT.get('smoke_test', {}).get('passed'), 'Ejecuta primero la PASO 08.'

def train_run(checkpoint_key, variant, training_seed):
    """Entrena una combinacion (checkpoint, variante, semilla) de principio a fin con
    `Trainer`: reinicia la semilla, carga el modelo preentrenado, aplica la variante
    (congela el encoder o no), entrena hasta `CFG['epochs'][variant]` epocas con
    parada temprana (`CFG['patience']`), selecciona el mejor checkpoint por F1 de
    validacion (`load_best_model_at_end`) y lo guarda en disco. `Trainer` reparte el
    lote entre las GPU visibles automaticamente si hay mas de una (no escribimos
    `DataParallel` a mano como en taller 1). Devuelve un diccionario con parametros,
    metricas finales de validacion, mejor epoca e historial de entrenamiento."""
    run_dir = RUN / checkpoint_key / variant / f'seed_{training_seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(training_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(training_seed)
    model = apply_variant(load_model(checkpoint_key), variant)
    parameters = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = TrainingArguments(
        output_dir=str(run_dir), seed=training_seed, data_seed=training_seed,
        num_train_epochs=CFG['epochs'][variant], learning_rate=CFG['learning_rate'][variant],
        per_device_train_batch_size=CFG['batch_size'], per_device_eval_batch_size=CFG['batch_size'],
        weight_decay=CFG['weight_decay'], warmup_steps=CFG['warmup_steps'],
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
        logging_strategy='epoch', report_to=[], disable_tqdm=False,
    )
    trainer = Trainer(model=model, args=args,
                       train_dataset=hf_datasets[checkpoint_key]['train'],
                       eval_dataset=hf_datasets[checkpoint_key]['validation'],
                       data_collator=collators[checkpoint_key],
                       processing_class=tokenizers[checkpoint_key],
                       compute_metrics=compute_metrics,
                       compute_loss_func=loss_fns[checkpoint_key],
                       callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['patience'])])
    print(f'{checkpoint_key}/{variant}, semilla {training_seed}. Dispositivo: {device}, '
          f'GPU visibles={torch.cuda.device_count()}, {trainable:,}/{parameters:,} parametros entrenables, '
          f'{len(hf_datasets[checkpoint_key]["train"])} bloques train, '
          f'{len(hf_datasets[checkpoint_key]["validation"])} bloques validacion.', flush=True)
    train_output = trainer.train()
    validation_metrics = trainer.evaluate()
    best_dir = run_dir / 'best_model'
    trainer.save_model(str(best_dir))
    best_epoch = next((row.get('epoch') for row in trainer.state.log_history
                       if row.get('step') == trainer.state.best_global_step), None)
    history = [{'step': row.get('step'), 'epoch': row.get('epoch'), 'train_loss': row.get('loss'),
               'eval_loss': row.get('eval_loss'), 'eval_f1': row.get('eval_f1')}
              for row in trainer.state.log_history if 'loss' in row or 'eval_f1' in row]
    # entity_spans/scores_from_counts ya devuelven float/int nativos de Python;
    # trainer.state.best_metric si puede ser un tensor 0-d, por eso el float() abajo.
    result = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': training_seed,
              'split_sha256': split_sha256, 'parameters': parameters, 'trainable_parameters': trainable,
              'checkpoint_dir': str(best_dir), 'best_epoch': best_epoch,
              'best_metric_f1': float(trainer.state.best_metric),
              'validation': {'micro': {'precision': float(validation_metrics['eval_precision']),
                                       'recall': float(validation_metrics['eval_recall']),
                                       'f1': float(validation_metrics['eval_f1']),
                                       'support': sum(int(validation_metrics.get(f'eval_support_{l}', 0)) for l in LABELS)},
                             'per_label': {l: {'f1': float(validation_metrics.get(f'eval_f1_{l}', 0.0)),
                                               'support': int(validation_metrics.get(f'eval_support_{l}', 0))} for l in LABELS},
                             'loss': float(validation_metrics['eval_loss'])},
              'train_runtime_seconds': train_output.metrics.get('train_runtime'),
              'epochs_completed': trainer.state.epoch, 'history': history}
    (run_dir / 'result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

REPORT['experiments'] = {}
for checkpoint_key in CFG['checkpoints']:
    REPORT['experiments'][checkpoint_key] = {}
    for variant in CFG['variants']:
        REPORT['experiments'][checkpoint_key][variant] = {}
        for training_seed in CFG['training_seeds']:
            result = train_run(checkpoint_key, variant, training_seed)
            REPORT['experiments'][checkpoint_key][variant][str(training_seed)] = result
            # Conservar avances si una ejecucion posterior falla.
            (RUN / 'experiments_progress.json').write_text(json.dumps(REPORT['experiments'], indent=2), encoding='utf-8')
n_runs = len(CFG['checkpoints']) * len(CFG['variants']) * len(CFG['training_seeds'])
print(f'PASO 09 OK. Terminaron {n_runs} entrenamientos. Ejecuta la PASO 10.')

### PASO 10 - Agregacion y tabla comparativa final

Resumimos los 12 entrenamientos sin volver a entrenar. Calculamos F1/precision/recall por semilla, medias y desviaciones estandar por combinacion (checkpoint x variante), la diferencia pareada `fine_tuned - frozen` dentro de cada checkpoint, la diferencia pareada `RoBERTa clinico - BETO` dentro de cada variante, y F1 por categoria. La tabla `aggregate` es la comparacion final que va en el informe.

In [ ]:
# PASO 10 - AGREGACION Y TABLA COMPARATIVA FINAL (4 combinaciones x 3 semillas)
experiments = REPORT.get('experiments', {})
assert set(experiments) == set(CFG['checkpoints']), 'Faltan checkpoints: completa la PASO 09.'
assert all(set(v) == set(CFG['variants']) for v in experiments.values()), 'Falta alguna variante.'
assert all(set(v[variant]) == {str(s) for s in CFG['training_seeds']}
          for v in experiments.values() for variant in CFG['variants']), 'Faltan semillas: completa la PASO 09.'

rows, label_rows = [], []
gold_support = None
for checkpoint_key in CFG['checkpoints']:
    for variant in CFG['variants']:
        for seed in CFG['training_seeds']:
            result = experiments[checkpoint_key][variant][str(seed)]
            assert result['seed'] == seed and result['split_sha256'] == REPORT['split']['sha256']
            metrics = result['validation']['micro']
            if gold_support is None: gold_support = metrics['support']
            assert metrics['support'] == gold_support, 'Las ejecuciones no evaluan el mismo gold.'
            rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                'parametros': result['parameters'], 'parametros_entrenables': result['trainable_parameters'],
                'mejor_epoca': result['best_epoch'], 'epocas_ejecutadas': result['epochs_completed'],
                'precision_pct': 100*metrics['precision'], 'recall_pct': 100*metrics['recall'],
                'f1_pct': 100*metrics['f1'], 'train_segundos': result['train_runtime_seconds']})
            for label in LABELS:
                m = result['validation']['per_label'][label]
                label_rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                                   'categoria': label, 'entidades_validacion': m['support'], 'f1_pct': 100*m['f1']})
comparison = pd.DataFrame(rows)
per_label = pd.DataFrame(label_rows)
assert (per_label.groupby('categoria')['entidades_validacion'].nunique() == 1).all()

# Desviacion estandar muestral (ddof=1): variabilidad de las semillas, no intervalo de confianza.
aggregate_rows = []
for (checkpoint_key, variant), group in comparison.groupby(['checkpoint', 'variante'], sort=False):
    row = {'checkpoint': checkpoint_key, 'variante': variant, 'n_semillas': len(group)}
    for metric in ['precision_pct', 'recall_pct', 'f1_pct', 'train_segundos']:
        row[metric + '_media'] = float(group[metric].mean())
        row[metric + '_std'] = float(group[metric].std(ddof=1))
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)

# Diferencia pareada 1: fine_tuned - frozen, dentro de cada checkpoint (misma semilla).
paired_variant_rows = []
for checkpoint_key in CFG['checkpoints']:
    sub = comparison[comparison.checkpoint == checkpoint_key]
    by_seed = sub.pivot(index='semilla', columns='variante', values='f1_pct')
    delta = by_seed['fine_tuned'] - by_seed['frozen']
    for seed, value in delta.items():
        paired_variant_rows.append({'checkpoint': checkpoint_key, 'semilla': seed,
                                    'f1_fine_tuned_pct': by_seed.loc[seed, 'fine_tuned'],
                                    'f1_frozen_pct': by_seed.loc[seed, 'frozen'], 'diferencia_pp': value})
paired_variant = pd.DataFrame(paired_variant_rows)

# Diferencia pareada 2: modelo clinico - BETO, dentro de cada variante (misma semilla).
paired_checkpoint_rows = []
for variant in CFG['variants']:
    sub = comparison[comparison.variante == variant]
    by_seed = sub.pivot(index='semilla', columns='checkpoint', values='f1_pct')
    delta = by_seed['clinical'] - by_seed['beto']
    for seed, value in delta.items():
        paired_checkpoint_rows.append({'variante': variant, 'semilla': seed,
                                       'f1_clinical_pct': by_seed.loc[seed, 'clinical'],
                                       'f1_beto_pct': by_seed.loc[seed, 'beto'], 'diferencia_pp': value})
paired_checkpoint = pd.DataFrame(paired_checkpoint_rows)

label_aggregate_rows = []
for checkpoint_key in CFG['checkpoints']:
    for variant in CFG['variants']:
        for label in LABELS:
            group = per_label[(per_label.checkpoint == checkpoint_key) & (per_label.variante == variant)
                              & (per_label.categoria == label)]
            label_aggregate_rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'categoria': label,
                'entidades_validacion': int(group.entidades_validacion.iloc[0]),
                'f1_media_pct': float(group.f1_pct.mean()), 'f1_std_pct': float(group.f1_pct.std(ddof=1))})
label_aggregate = pd.DataFrame(label_aggregate_rows)

for name, table in [('comparison_runs', comparison), ('comparison_aggregate', aggregate),
                    ('comparison_paired_variant', paired_variant), ('comparison_paired_checkpoint', paired_checkpoint),
                    ('comparison_per_label_runs', per_label), ('comparison_per_label_aggregate', label_aggregate)]:
    table.to_csv(RUN / f'{name}.csv', index=False)

REPORT['comparison'] = {'per_run': rows, 'aggregate': aggregate_rows,
    'paired_variant': paired_variant_rows, 'paired_checkpoint': paired_checkpoint_rows,
    'per_label_aggregate': label_aggregate_rows,
    'note': 'Media y desviacion muestral entre semillas de entrenamiento, con una particion fija. No es validacion cruzada ni una prueba de significancia estadistica.'}
REPORT['elapsed_seconds_since_cell02'] = time.perf_counter() - START
REPORT['status'] = 'COMPARACION_12_ENTRENAMIENTOS_COMPLETADA'
REPORT['metric_scope'] = 'Entidades exactas BIO (entity_spans/scores_from_counts, misma funcion que taller 1).'
(RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')

print('TABLA COMPARATIVA FINAL (F1 medio +/- desviacion estandar, en %):')
print(aggregate.round(3).to_string(index=False))
print()
print('DIFERENCIA fine_tuned - frozen (por checkpoint):')
print(paired_variant.round(3).to_string(index=False))
print()
print('DIFERENCIA clinical - beto (por variante):')
print(paired_checkpoint.round(3).to_string(index=False))
print()
print('F1 POR CATEGORIA:')
print(label_aggregate.round(3).to_string(index=False))
print()
print('Archivos guardados en:', RUN)
print('Pega la tabla comparativa (aggregate) en el informe final de este cuaderno.')

### PASO 11 - Evaluacion final en test (opcional, desactivada)

Igual que en taller 1: el conjunto de test se mantiene reservado por defecto. Solo se evalua si activas `run_final_test=True` y eliges explicitamente `final_test_checkpoint`, `final_test_variant` y `final_test_seed` despues de revisar la tabla comparativa de la PASO 10 -- nunca se elige automaticamente "la mejor" combinacion.

In [ ]:
# PASO 11 - TEST FINAL OPCIONAL DE UNA COMBINACION ELEGIDA; DESACTIVADO
if not CFG['run_final_test']:
    print('PASO 11 OMITIDA: test reservado. Comparacion terminada en la PASO 10.')
elif CFG['fraction'] != 1.0:
    raise RuntimeError('Test bloqueado: completa primero el entrenamiento con fraction=1.0.')
elif CFG['final_test_checkpoint'] not in CFG['checkpoints']:
    raise RuntimeError(f'Selecciona explicitamente final_test_checkpoint entre {list(CFG["checkpoints"])}.')
elif CFG['final_test_variant'] not in CFG['variants']:
    raise RuntimeError(f'Selecciona explicitamente final_test_variant entre {CFG["variants"]}.')
elif CFG['final_test_seed'] not in CFG['training_seeds']:
    raise RuntimeError('Selecciona explicitamente final_test_seed entre las semillas entrenadas.')
else:
    checkpoint_key = CFG['final_test_checkpoint']
    variant = CFG['final_test_variant']
    seed = CFG['final_test_seed']
    result = REPORT['experiments'][checkpoint_key][variant][str(seed)]
    best_dir = result['checkpoint_dir']
    model = AutoModelForTokenClassification.from_pretrained(best_dir).to(device)
    test_examples, test_stats = make_examples(checkpoint_key, official_test_ids, df_test, 'test')
    test_dataset = Dataset.from_list(test_examples)
    test_args = TrainingArguments(output_dir=str(RUN / 'final_test_tmp'),
                                  per_device_eval_batch_size=CFG['batch_size'], report_to=[])
    test_trainer = Trainer(model=model, args=test_args, data_collator=collators[checkpoint_key],
                           processing_class=tokenizers[checkpoint_key], compute_metrics=compute_metrics,
                           compute_loss_func=loss_fns[checkpoint_key])
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset)
    # entity_spans/scores_from_counts ya devuelven float/int nativos de Python.
    final_test = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': seed,
                 'metrics': {'loss': float(test_metrics['eval_loss']),
                            'micro': {'precision': float(test_metrics['eval_precision']),
                                     'recall': float(test_metrics['eval_recall']), 'f1': float(test_metrics['eval_f1'])},
                            'per_label': {l: {'f1': float(test_metrics.get(f'eval_f1_{l}', 0.0)),
                                              'support': int(test_metrics.get(f'eval_support_{l}', 0))} for l in LABELS}},
                 'alignment': test_stats, 'checkpoint_dir': best_dir, 'metric_scope': REPORT['metric_scope']}
    (RUN / f'final_test_{checkpoint_key}_{variant}_seed_{seed}.json').write_text(json.dumps(final_test, indent=2), encoding='utf-8')
    print('RESULTADO_TEST_FINAL', json.dumps(final_test, indent=2))

### PASO 12 - Identificacion interactiva en un texto nuevo

Cargamos el checkpoint elegido con `pipeline("token-classification", aggregation_strategy="first")` de Hugging Face, que tokeniza, predice y agrupa subwords en entidades automaticamente -- mucho mas simple que la reconstruccion manual de taller 1.

In [ ]:
# PASO 12 - IDENTIFICACION INTERACTIVA EN UN TEXTO NUEVO
from transformers import pipeline

TEXT_TO_IDENTIFY = ("Paciente con cefalea intensa y fiebre. Se indico tratamiento "
                    "con paracetamol y se solicito una resonancia magnetica.")
IDENTIFICATION_CHECKPOINT = 'clinical'   # 'beto' o 'clinical'
IDENTIFICATION_VARIANT = 'fine_tuned'    # 'frozen' o 'fine_tuned'
IDENTIFICATION_SEED = 42                 # una semilla de CFG['training_seeds']

if not isinstance(TEXT_TO_IDENTIFY, str) or not TEXT_TO_IDENTIFY.strip():
    raise ValueError('TEXT_TO_IDENTIFY debe ser una cadena no vacia.')
if IDENTIFICATION_CHECKPOINT not in CFG['checkpoints']:
    raise ValueError(f'IDENTIFICATION_CHECKPOINT debe pertenecer a {list(CFG["checkpoints"])}.')
if IDENTIFICATION_VARIANT not in CFG['variants']:
    raise ValueError(f'IDENTIFICATION_VARIANT debe pertenecer a {CFG["variants"]}.')
if IDENTIFICATION_SEED not in CFG['training_seeds']:
    raise ValueError(f'IDENTIFICATION_SEED debe pertenecer a {CFG["training_seeds"]}.')
experiments = REPORT.get('experiments', {})
combo = experiments.get(IDENTIFICATION_CHECKPOINT, {}).get(IDENTIFICATION_VARIANT, {})
if str(IDENTIFICATION_SEED) not in combo:
    raise RuntimeError('No existe el resultado de esa combinacion; ejecuta primero la PASO 09.')

result = combo[str(IDENTIFICATION_SEED)]
checkpoint_dir = result['checkpoint_dir']
identifier = pipeline('token-classification', model=checkpoint_dir, tokenizer=tokenizers[IDENTIFICATION_CHECKPOINT],
                      aggregation_strategy='first', device=0 if torch.cuda.is_available() else -1)
raw_entities = identifier(TEXT_TO_IDENTIFY)
entities = [{'text': e['word'], 'label': e['entity_group'], 'start': int(e['start']), 'end': int(e['end']),
            'confidence_mean': round(float(e['score']), 4)} for e in raw_entities]
identification = {'text': TEXT_TO_IDENTIFY, 'checkpoint': IDENTIFICATION_CHECKPOINT,
                  'variant': IDENTIFICATION_VARIANT, 'training_seed': IDENTIFICATION_SEED,
                  'checkpoint_dir': checkpoint_dir, 'entities': entities,
                  'warning': 'La confianza no esta calibrada; las predicciones no sustituyen evaluacion clinica.'}
print('========== IDENTIFICACION ==========', flush=True)
print(json.dumps(identification, ensure_ascii=False, indent=2), flush=True)
print('=====================================', flush=True)
print(f"Entidades detectadas: {len(entities)}", flush=True)

# Informe final (plantilla) - NER clinico en espanol con Transformers

## Objetivo y diseno

Comparamos cuatro combinaciones para NER clinico en espanol: dos checkpoints preentrenados (`dccuchile/bert-base-spanish-wwm-cased`, "BETO", BERT generico; y `PlanTL-GOB-ES/bsc-bio-ehr-es`, un RoBERTa preentrenado en textos biomedicos/clinicos en espanol) por dos variantes de fine-tuning (`frozen`: solo se entrena el cabezal; `fine_tuned`: todos los pesos). Usamos la misma particion de documentos de taller 1 (`split_seed=42`, 600 documentos de entrenamiento, 150 de validacion, 250 de test reservados) y las mismas tres semillas de entrenamiento (42, 123, 2026), para que el F1 final sea comparable entre los dos talleres.

**Esta celda es una plantilla.** Este cuaderno no se ha ejecutado en el entorno donde se escribio (no hay GPU disponible aqui). Ejecuta las PASO 01 a 10 en Kaggle (2x T4) o en Colab (ajustando `requested_gpus=1`) y reemplaza los valores entre `<>` con la salida real de la PASO 10 antes de entregar el informe.

## Resultado de validacion multisemilla

| Checkpoint | Variante | F1 medio | Desv. estandar | Precision media | Recall medio |
|---|---|---:|---:|---:|---:|
| BETO | frozen | `<F1>` % | `<STD>` % | `<P>` % | `<R>` % |
| BETO | fine_tuned | `<F1>` % | `<STD>` % | `<P>` % | `<R>` % |
| RoBERTa clinico | frozen | `<F1>` % | `<STD>` % | `<P>` % | `<R>` % |
| RoBERTa clinico | fine_tuned | `<F1>` % | `<STD>` % | `<P>` % | `<R>` % |

*(Copiar de la tabla `aggregate` que imprime la PASO 10.)*

## Comparaciones pareadas

- **fine_tuned vs frozen** (dentro de cada checkpoint, misma semilla): pegar la tabla `comparison_paired_variant` de la PASO 10.
- **RoBERTa clinico vs BETO** (dentro de cada variante, misma semilla): pegar la tabla `comparison_paired_checkpoint` de la PASO 10.

## F1 medio por categoria

Pegar aqui la tabla `comparison_per_label_aggregate` de la PASO 10 (una fila por checkpoint x variante x categoria).

## Comparacion con taller 1 (Bi-LSTM entrenada desde cero)

| Modelo | F1 medio en validacion |
|---|---:|
| Bi-LSTM con POS (taller 1) | 38.96 % |
| Bi-LSTM sin POS (taller 1) | 37.13 % |
| BETO, mejor variante (taller 2) | `<F1>` % |
| RoBERTa clinico, mejor variante (taller 2) | `<F1>` % |

## Evaluacion final en test

`<Completar solo si se activo la PASO 11: variante, checkpoint, semilla y metricas de RESULTADO_TEST_FINAL.>`

## Limitaciones y alcance

Igual que en taller 1: el esquema BIO es plano (las anotaciones anidadas o solapadas se excluyen y se contabilizan en `alignment_*.json`); la comparacion usa una sola particion fija y tres semillas, por lo que la desviacion estandar describe variabilidad de entrenamiento, no una prueba de significancia estadistica. La variante `frozen` congela el encoder completo (no capas individuales), asi que funciona como una cota inferior de lo que el mismo checkpoint podria lograr con fine-tuning parcial. La confianza de la PASO 12 no esta calibrada y las predicciones no sustituyen una decision clinica.

## Conclusion

`<Completar despues de ejecutar la PASO 10: que combinacion (checkpoint + variante) recomiendan como configuracion base, y como se compara contra la Bi-LSTM de taller 1?>`

Artefactos: revisa la carpeta `RUN` (impresa en cada PASO, dentro de `spaccc_transformers/`) para `summary.json`, las tablas `comparison_*.csv` y los checkpoints `best_model/` de cada una de las 12 combinaciones.